In [ ]:
#webtraffic
import pandas as pd

def process_web_traffic(input_file='web_traffic.csv', output_file='web_traffic.csv'):
    # Đọc dữ liệu thô
    df = pd.read_csv(input_file)

    df_clean = pd.DataFrame()

    # 1. Traffic_ID
    if 'traffic_id' in df.columns:
        df_clean['Traffic_ID'] = df['traffic_id'].astype(str).str.strip()
    elif 'Traffic_ID' in df.columns:
        df_clean['Traffic_ID'] = df['Traffic_ID'].astype(str).str.strip()
    else:
        df_clean['Traffic_ID'] = ['TRF-' + str(i + 1).zfill(7) for i in range(len(df))]

    # 2. Session_Date
    if 'session_date' in df.columns:
        df_clean['Session_Date'] = pd.to_datetime(df['session_date'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
    elif 'date' in df.columns:
        df_clean['Session_Date'] = pd.to_datetime(df['date'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
    else:
        df_clean['Session_Date'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')

    # 3. Page_Visited
    if 'page_visited' in df.columns:
        df_clean['Page_Visited'] = df['page_visited'].astype(str).str.strip()
    elif 'traffic_source' in df.columns:
        df_clean['Page_Visited'] = df['traffic_source'].astype(str).str.strip().str.replace('_', ' ').str.title()
    else:
        df_clean['Page_Visited'] = 'Home Page'

    # 4. Time_Spent (giây)
    if 'time_spent' in df.columns:
        df_clean['Time_Spent'] = pd.to_numeric(df['time_spent'], errors='coerce').fillna(0.0).round(2)
    elif 'avg_session_duration_sec' in df.columns:
        df_clean['Time_Spent'] = pd.to_numeric(df['avg_session_duration_sec'], errors='coerce').fillna(0.0).round(2)
    else:
        df_clean['Time_Spent'] = 0.0

    # 5. Browser_Info
    if 'browser_info' in df.columns:
        df_clean['Browser_Info'] = df['browser_info'].astype(str).str.strip()
    elif 'Browser_Info' in df.columns:
        df_clean['Browser_Info'] = df['Browser_Info'].astype(str).str.strip()
    else:
        df_clean['Browser_Info'] = 'Chrome / Windows'

    # 6. Customer_ID
    if 'customer_id' in df.columns:
        df_clean['Customer_ID'] = df['customer_id'].astype(str).str.strip()
    elif 'Customer_ID' in df.columns:
        df_clean['Customer_ID'] = df['Customer_ID'].astype(str).str.strip()
    else:
        df_clean['Customer_ID'] = 'ANONYMOUS'

    # Sắp xếp đúng cấu trúc 6 cột theo chuẩn WEB_TRAFFIC
    cols = ['Traffic_ID', 'Session_Date', 'Page_Visited', 'Time_Spent', 'Browser_Info', 'Customer_ID']
    traffic_table = df_clean[cols].drop_duplicates(subset=['Traffic_ID']).dropna(subset=['Traffic_ID'])

    # Xuất file CSV
    traffic_table.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng WEB_TRAFFIC: {output_file} ({len(traffic_table):,} dòng)")

if __name__ == '__main__':
    process_web_traffic()

In [ ]:
import pandas as pd

def process_traffic_sources(input_file='web_traffic.csv', output_file='traffic_sources.csv'):
    df = pd.read_csv(input_file)

    source_col = 'traffic_source' if 'traffic_source' in df.columns else 'Page_Visited'

    sources = pd.DataFrame(df[source_col].unique(), columns=['Source_Code'])
    sources['Source_Code'] = sources['Source_Code'].astype(str).str.strip().str.lower()
    sources = sources.drop_duplicates().reset_index(drop=True)

    sources['Source_ID'] = ['SRC_' + str(i + 1).zfill(3) for i in range(len(sources))]
    sources['Source_Name'] = sources['Source_Code'].str.replace('_', ' ').str.title()

    cols = ['Source_ID', 'Source_Code', 'Source_Name']
    sources_df = sources[cols]

    sources_df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng TRAFFIC_SOURCES: {output_file} ({len(sources_df):,} dòng)")

if __name__ == '__main__':
    process_traffic_sources()

In [ ]:
import pandas as pd

def process_daily_traffic(input_file='web_traffic.csv', output_file='daily_traffic_summary.csv'):
    df = pd.read_csv(input_file)

    date_col = 'date' if 'date' in df.columns else 'Session_Date'
    sess_col = 'sessions' if 'sessions' in df.columns else 'Traffic_ID'
    visitor_col = 'unique_visitors' if 'unique_visitors' in df.columns else 'Customer_ID'
    views_col = 'page_views' if 'page_views' in df.columns else 'Page_Visited'
    dur_col = 'avg_session_duration_sec' if 'avg_session_duration_sec' in df.columns else 'Time_Spent'

    df['date_clean'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d')

    daily_summary = df.groupby('date_clean').agg(
        Total_Sessions=(sess_col, 'sum' if 'sessions' in df.columns else 'count'),
        Total_Visitors=(visitor_col, 'sum' if 'unique_visitors' in df.columns else 'nunique'),
        Total_Page_Views=(views_col, 'sum' if 'page_views' in df.columns else 'count'),
        Avg_Duration_Sec=(dur_col, 'mean')
    ).reset_index()

    daily_summary.rename(columns={'date_clean': 'Date'}, inplace=True)
    daily_summary['Avg_Duration_Sec'] = daily_summary['Avg_Duration_Sec'].round(2)

    daily_summary.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng DAILY_TRAFFIC_SUMMARY: {output_file} ({len(daily_summary):,} dòng)")

if __name__ == '__main__':
    process_daily_traffic()